# KNN (first attempt)

## Using code from Prince's in-class example + lab 9

Goal: predict if a new loan will default

How-to:
1. Measure distance to other loans in training data
2. Find the K closest (neighbors) loans
3. Let those loans "vote" to decide on how to predict the behavior of this new loan

Steps for building the model:
- Split off the test set and wait to use it until final evaluation at the end
- Do CV on training dataset for each potential model choice (values of K, and measure of similarity (Euclidean vs. Manhattan))
- Find the choices that have the best CV score (highest balanced accuracy/lowest classification error)
- Refit this model on all of the training data
- Check final model's performance using the test set

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix
)

In [2]:
loan_df_clean = pd.read_csv('Loan_default_cleaned.csv')
loan_df_clean.head()

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DebtToIncomeRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default,LoanToIncomeRatio
0,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0,0.588262
1,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0,2.467481
2,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1,1.534154
3,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0,1.412638
4,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0,0.447179


In [3]:
# using predictors from Kaitlyn's logisitic regression model
y = loan_df_clean['Default']

# save the predictors
X = loan_df_clean[['Age', 'Income', 'LoanAmount', 'CreditScore',
       'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm',
       'DebtToIncomeRatio', 'Education', 'EmploymentType', 'MaritalStatus',
       'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner', 'LoanToIncomeRatio']]

# save the categorical variables
cats = ['Education', 'EmploymentType','MaritalStatus',
        'HasMortgage', 'HasDependents','LoanPurpose','HasCoSigner']

# save the numerical variables
nums = ['Age', 'Income', 'LoanAmount',   'CreditScore',
        'MonthsEmployed','NumCreditLines','InterestRate', 'LoanTerm',
        'DebtToIncomeRatio', 'LoanToIncomeRatio']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=6021, stratify=y) # stratify because we have class imbalance

display(pd.DataFrame({
    'Split': ['Train (used for CV)', 'Test'],
    'Rows': [len(y_train), len(y_test)],
    'Default rate': [y_train.mean(), y_test.mean()]
}))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) # for 5-fold cross-validation

,Split,Rows,Default rate
0,Train (used for CV),191510,0.116130
1,Test,63837,0.116124


# Preprocessing and pipeline

In [6]:
# create the order of the pipeline
preprocess = ColumnTransformer(transformers=[
    ("encoder", OneHotEncoder(drop='if_binary', handle_unknown='ignore'), cats), 
    # got rid of drop='first', since KNN does not have multicollinearity issues (no coefficients)
    # drop='if_binary' keeps binary features from counting double in the distance.
    ("numeric", StandardScaler(), nums) # distance is scale-sensitive, so v important to standardize predictors
])

pipe = Pipeline([
    ("preprocess", preprocess),
    ("knn", KNeighborsClassifier())
])

# Tuning: choosing K and distance metric

In [24]:
param_grid = {
    "knn__n_neighbors": range(1, 41, 2),
     # number of neighbors: outcome is binary, we use odd numbers to avoid ties
    "knn__metric": ["euclidean", "manhattan"], # "similarity" measure
    "knn__weights": ["uniform", "distance"] # voting rule
}

grid = GridSearchCV(
    pipe, param_grid, cv=cv, scoring="balanced_accuracy",
    return_train_score=True,
    n_jobs=-1, # use all available CPU cores (so fits run in parallel AKA faster)
    error_score="raise" # if any fit fails, stop and show error
)
grid.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'knn__metric': ['euclidean', 'manhattan'], 'knn__n_neighbors': range(1, 41, 2), 'knn__weights': ['uniform', 'distance']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'balanced_accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"error_score error_score: 'raise' or numeric, default=np.nanValue to assign to the score if an error occurs in estimator fitting.If set to 'raise', the error is raised. If a numeric value is given,FitFailedWarning is raised. This parameter does not affect the refitstep, which will always raise the error.",'raise'
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_i

In [33]:
results_df = pd.DataFrame(grid.cv_results_)
results_df["k"] = results_df["param_knn__n_neighbors"].astype(int)
results_df["metric"] = results_df["param_knn__metric"]
results_df["cv_balanced_error"] = 1 - results_df["mean_test_score"]
results_df["train_balanced_error"] = 1 - results_df["mean_train_score"]
results_df["weights"] = results_df["param_knn__weights"]

best_k = grid.best_params_["knn__n_neighbors"]
best_metric = grid.best_params_["knn__metric"]
best_error = 1 - grid.best_score_
best_weights = grid.best_params_["knn__weights"]

print("Best K:", best_k)
print("Best distance:", best_metric)
print("Best voting rule (weights):", best_weights)
print(f"Best CV balanced classification error: {best_error:.3f}")

Best K: 1
Best distance: euclidean
Best voting rule (weights): uniform
Best CV balanced classification error: 0.456


In [34]:
# create text label for each metric/voting combo so each line can be plotted separately
results_df["configuration"] = results_df["metric"] + " + " + results_df["weights"] 

fig = px.line(
    results_df.sort_values("k"), x="k", y="cv_balanced_error", color="configuration", markers=True,
    labels={"k": "Number of neighbors (K)",
            "cv_balanced_error": "CV balanced classification error",
            "configuration": "Metric / voting"},
    title=f"Choosing K: {best_metric}, {best_weights} votes, K = {best_k}"
)
fig.add_scatter(
    x=[best_k], y=[best_error], mode="markers+text",
    text=[f"Best K = {best_k}"], textposition="top center", name="Selected model"
)
fig.update_layout(hovermode="x unified")
fig.show()

- CV balanced error was lowest at K=1 using Euclidean distance (0.455). It looks like Euclidean beats Manhattan at every K. This is CV balanced error is only slightly better than chance (0.5) with about 11% defaults; larger K lets non defaulters dominate the vote, so balanced error rises toward 0.5 as K grows.

In [35]:
curve = results_df.loc[
(results_df["metric"] == "euclidean") &
(results_df["weights"] == "uniform"),
["k", "train_balanced_error", "cv_balanced_error"]
].sort_values("k")
display(curve.round(4))
curve_plot = curve.rename(columns={
"train_balanced_error": "Training folds",
"cv_balanced_error": "Held-out CV folds"
})
fig = px.line(
curve_plot, x="k", y=["Training folds", "Held-out CV folds"],
markers=True, labels={"k": "Number of neighbors (K)",
"value": "Mean balanced error", "variable": "Data"},
title="Changing K: standardized Euclidean distance, uniform votes"
)
fig.show()

,k,train_balanced_error,cv_balanced_error
0,1,0.0000,0.4557
2,3,0.3430,0.4589
4,5,0.4114,0.4649
6,7,0.4385,0.4699
8,9,0.4529,0.4736
10,11,0.4611,0.4764
12,13,0.4664,0.4777
14,15,0.4706,0.4790
16,17,0.4734,0.4805
18,19,0.4755,0.4812


In [ ]:
# code so that we can use the best model without having to run the grid search again
#final_model = Pipeline([
#    ("preprocess", preprocess),
#   ("knn", KNeighborsClassifier(n_neighbors=1, metric="euclidean", weights="uniform"))
#])
#final_model.fit(X_train, y_train)
#y_pred = final_model.predict(X_test)

In [ ]:
pipe2 = grid.best_estimator_
y_pred = pipe2.predict(X_test)

In [9]:
acc = accuracy_score(y_test, y_pred)
bal_acc = balanced_accuracy_score(y_test, y_pred)

print(f"Accuracy: {acc:.3f}")
print(f"Classification error (1 - accuracy): {1 - acc:.3f}")
print(f"Balanced accuracy: {bal_acc:.3f}")
print(f"Balanced classification error: {1 - bal_acc:.3f}")
print(f"Precision for Default: {precision_score(y_test, y_pred, zero_division=0):.3f}")
print(f"Recall for Default: {recall_score(y_test, y_pred, zero_division=0):.3f}")
print(f"F1 for Default: {f1_score(y_test, y_pred, zero_division=0):.3f}")

Accuracy: 0.816
Classification error (1 - accuracy): 0.184
Balanced accuracy: 0.545
Balanced classification error: 0.455
Precision for Default: 0.199
Recall for Default: 0.192
F1 for Default: 0.196


In [10]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

fig = px.imshow(
    cm, text_auto=True, color_continuous_scale="Blues",
    x=["No default", "Default"], y=["No default", "Default"],
    labels={"x": "Predicted", "y": "Actual", "color": "Borrowers"},
    title="Test-set confusion matrix: selected kNN"
)
fig.show()